# 📖 Lab 3: Distributed Rate Limiting with Redis

In Labs 1 and 2, our rate limiters lived in a single Python process. That works for one server, but what happens when you have **multiple servers** behind a load balancer?

Each server only sees its own traffic — if user Alice hits Server A 50 times and Server B 50 times, each server thinks she's at 50, but she's actually at **100**!

We need a **shared counter** that all servers can access. That's where **Redis** comes in — a blazing fast in-memory database perfect for this job.

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand why local rate limiters fail** in distributed systems
2. **Use Redis as a shared rate limit store** that all servers can access
3. **Implement atomic operations with Lua scripts** to prevent race conditions
4. **Build a production-ready distributed token bucket** that works across multiple servers

## 🛠️ Setup

### 1. Start the Docker containers

Open a terminal in the `06-system-designs/rate-limiter/` directory and run:

```bash
docker compose up -d
```

This starts Redis on **port 6381** and RedisInsight on **port 5541**.

### 2. Select the correct Jupyter kernel

In the top-right corner of this notebook, click the **kernel picker** and select `.venv (Python)` or `rate-limiter`.

> 💡 If the kernel doesn't appear, reload the VS Code window: `Cmd+Shift+P` → **"Reload Window"**

### 3. Install dependencies

If you haven't already:

```bash
cd 06-system-designs/rate-limiter
uv venv
source .venv/bin/activate
uv sync
```

> ⚠️ Redis is running on port **6381** (not the default 6379) to avoid conflicts with any local Redis instance.

In [ ]:
import redis
import time

# Connect to Redis (running in Docker on port 6381)
r = redis.Redis(host="localhost", port=6381, decode_responses=True)

# Test the connection
print(f"Connected to Redis: {r.ping()}")
print(f"Redis info: {r.info('server')['redis_version']}")

# Clean up any leftover keys from previous runs
for key in r.keys("ratelimit:*"):
    r.delete(key)
print("Cleaned up old rate limit keys")

## 🤔 The Problem — Why Local Counters Fail

When you run **one server**, a local counter (like a Python dictionary) works great. But most real applications run on **multiple servers** behind a load balancer.

Here's the problem:

```
                    ┌─────────────┐
                    │ Load        │
     Alice ──────>  │ Balancer    │
    (100 req)       └──────┬──────┘
                     ┌─────┴─────┐
                     │           │
              ┌──────▼──────┐ ┌─▼──────────┐
              │ Server A    │ │ Server B    │
              │ counter: 50 │ │ counter: 50 │
              │ limit: 100  │ │ limit: 100  │
              │ "She's fine!"│ │ "She's fine!"│
              └─────────────┘ └─────────────┘
              
     Reality: Alice made 100 requests total — should be at the limit!
```

Each server has its **own counter** and only sees **half** the traffic. Alice bypasses the rate limit!

### ✅ The Solution: A Shared Counter in Redis

With Redis, **all servers share one counter**:

```
              ┌──────────────┐   ┌──────────────┐
              │ Server A     │   │ Server B     │
              └──────┬───────┘   └──────┬───────┘
                     │                   │
                     └───────┬───────────┘
                             │
                      ┌──────▼──────┐
                      │   Redis     │
                      │ alice: 100  │
                      │ "She's at   │
                      │  the limit!"│
                      └─────────────┘
```

Redis is an **in-memory database** — reads and writes take less than 1 millisecond. That makes it perfect for rate limiting where every request needs to be checked.

## 🧪 Approach 1 — Simple Redis Counter (Has a Bug!)

Let's start with the simplest thing that could work: use Redis `INCR` to count requests.

**The idea:**
1. For each request, read the current count from Redis
2. If it's under the limit, increment it
3. Set an expiration so the key auto-deletes after the window

This *works* — but it has a subtle **race condition**. Let's build it first, then see the bug.

In [ ]:
import redis
import time

r = redis.Redis(host="localhost", port=6381, decode_responses=True)

def simple_rate_limit(client_id: str, max_requests: int, window_seconds: int) -> bool:
    """Simple rate limiter using Redis INCR.
    
    This works but has a race condition — can you spot it?
    Hint: what happens between the GET and the INCR?
    """
    key = f"ratelimit:simple:{client_id}"
    
    # Step 1: Get the current count
    current = r.get(key)
    
    # Step 2: Check if over the limit
    if current is not None and int(current) >= max_requests:
        return False  # Rate limited!
    
    # Step 3: Increment the counter and set expiration
    # ⚠️ BUG: Between Step 1 and Step 3, another server could also read the same value!
    pipe = r.pipeline()
    pipe.incr(key)
    pipe.expire(key, window_seconds)
    pipe.execute()
    
    return True  # Request allowed

# --- Test it ---
print("Simple Redis rate limiter: 5 requests per 10 seconds")
r.delete("ratelimit:simple:alice")

for i in range(7):
    allowed = simple_rate_limit("alice", max_requests=5, window_seconds=10)
    status = "✅ ALLOWED" if allowed else "❌ DENIED"
    count = r.get("ratelimit:simple:alice")
    print(f"  Request {i+1}: {status}  (Redis counter: {count})")

## ⚠️ The Race Condition Problem

The simple approach above has a bug called **TOCTOU** — **Time Of Check to Time Of Use**.

Here's what can happen when two servers process requests at the same time:

| Step | Server A | Server B | Redis Counter |
|------|----------|----------|---------------|
| 1 | Reads counter: **99** | | 99 |
| 2 | | Reads counter: **99** | 99 |
| 3 | Thinks "99 < 100 → allow!" | | 99 |
| 4 | | Thinks "99 < 100 → allow!" | 99 |
| 5 | Increments → **100** | | 100 |
| 6 | | Increments → **101** | **101** |

Both servers allowed the request, but the limit was 100! The counter ended up at **101**.

The problem: the **check** (reading the counter) and the **action** (incrementing it) happen in **separate steps**. Between those steps, another server can sneak in.

Let's prove this happens in practice:

In [ ]:
import threading
import redis

r = redis.Redis(host="localhost", port=6381, decode_responses=True)

LIMIT = 10
GATEWAYS = 30      # concurrent "servers", each with its own Redis connection
PER_GATEWAY = 3    # requests each one sends


def race_round() -> int:
    """Hammer the *actual* simple_rate_limit() from above. Returns allowed count."""
    r.delete("ratelimit:simple:alice")
    allowed = []
    bookkeeping = threading.Lock()

    def gateway():
        for _ in range(PER_GATEWAY):
            ok = simple_rate_limit("alice", max_requests=LIMIT, window_seconds=30)
            with bookkeeping:
                allowed.append(ok)

    threads = [threading.Thread(target=gateway) for _ in range(GATEWAYS)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return sum(allowed)


print("=== Race Condition Demo ===")
print(f"Limit: {LIMIT}. {GATEWAYS} gateways x {PER_GATEWAY} requests = "
      f"{GATEWAYS * PER_GATEWAY} requests, all at once.")
print("Nothing is staged here — this is the same simple_rate_limit() defined above.\n")

rounds = [race_round() for _ in range(3)]
for i, n in enumerate(rounds, 1):
    print(f"  Round {i}: {n} allowed  (limit was {LIMIT}) "
          f"→ {n - LIMIT} over" if n > LIMIT else f"  Round {i}: {n} allowed  ✅ exact")

worst = max(rounds)
print(f"\n  Worst case: {worst} allowed against a limit of {LIMIT} "
      f"— {100 * (worst - LIMIT) / LIMIT:.0f}% over.")
assert worst > LIMIT, "expected the TOCTOU race to overshoot the limit"

print("""
  Every one of those extra requests is a gateway that read the counter,
  got a value under the limit, and then incremented it — while other
  gateways were doing exactly the same thing with the same stale value.
  The gap between GET and INCR is the entire bug. Notice it needs no
  artificial sleep to reproduce: a Redis round-trip is microseconds, but
  so is the window, and 30 threads find it every time.""")

## 💡 The Solution — Redis Lua Scripts (Atomic Operations)

How do we fix this? We need the **read + check + update** to happen as a **single, uninterruptible operation**.

Redis supports **Lua scripts** — small programs that run **inside Redis itself**. The key property:

> 🔒 **A Lua script is atomic.** While it runs, no other Redis command can execute. The entire script completes before anything else happens.

Think of it like putting the entire read-check-update logic inside a single lock:

```
Without Lua (race condition possible):
  Server A: READ ───────────── CHECK ── WRITE
  Server B:      READ ─ CHECK ─── WRITE
                 ↑ Both read the same value!

With Lua (atomic — no race condition):
  Server A: [READ + CHECK + WRITE]  ← runs as ONE operation
  Server B:                         [READ + CHECK + WRITE]
                                    ↑ Waits until A finishes
```

Now let's build a **token bucket** rate limiter using a Lua script.

In [ ]:
import redis
import time

r = redis.Redis(host="localhost", port=6381, decode_responses=True)

# ===========================================================================
# This Lua script runs ATOMICALLY inside Redis.
# No other command can run between the read and the write — no race conditions!
# ===========================================================================
TOKEN_BUCKET_LUA = """
local key = KEYS[1]
local max_tokens = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])
local now = tonumber(ARGV[3])

-- Read current bucket state from Redis
local bucket = redis.call('HMGET', key, 'tokens', 'last_refill')
local tokens = tonumber(bucket[1])
local last_refill = tonumber(bucket[2])

-- Initialize if this is a new bucket (first request from this client)
if tokens == nil then
    tokens = max_tokens
    last_refill = now
end

-- Calculate how many tokens to add since the last refill
local elapsed = now - last_refill
local new_tokens = elapsed * refill_rate
tokens = math.min(max_tokens, tokens + new_tokens)

-- Try to consume one token
local allowed = 0
if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
end

-- Save the updated state (still inside the atomic script!)
redis.call('HMSET', key, 'tokens', tostring(tokens), 'last_refill', tostring(now))
-- Auto-delete inactive buckets after 1 hour to prevent memory leaks
redis.call('EXPIRE', key, 3600)

-- Return: [allowed (1 or 0), remaining tokens]
return {allowed, math.floor(tokens)}
"""

# Register the script with Redis (returns a SHA hash for fast repeated execution)
script_sha = r.script_load(TOKEN_BUCKET_LUA)

def distributed_token_bucket(client_id: str, max_tokens: int = 5, refill_rate: float = 1.0) -> dict:
    """Check rate limit using an atomic Lua script in Redis.
    
    This is safe for distributed systems — no race conditions!
    
    Args:
        client_id: Who is making the request (e.g., "alice")
        max_tokens: Maximum burst size (bucket capacity)
        refill_rate: Tokens added per second (sustained rate)
    
    Returns:
        dict with 'allowed' (bool) and 'remaining' (int)
    """
    key = f"ratelimit:token:{client_id}"
    now = time.time()
    
    # Execute the Lua script atomically in Redis
    result = r.evalsha(script_sha, 1, key, max_tokens, refill_rate, now)
    
    return {
        "allowed": bool(result[0]),
        "remaining": int(result[1]),
    }

# --- Clean up and test ---
for key in r.keys("ratelimit:token:*"):
    r.delete(key)

print("=== Distributed Token Bucket (Redis + Lua) ===")
print("Config: max_tokens=5, refill_rate=1/sec")
print()

# Send 7 requests quickly — only 5 should be allowed
for i in range(7):
    result = distributed_token_bucket("alice", max_tokens=5, refill_rate=1)
    status = "✅ ALLOWED" if result["allowed"] else "❌ DENIED"
    print(f"  Request {i+1}: {status}  (remaining: {result['remaining']})")

# Wait for tokens to refill
print("\nWaiting 3 seconds for tokens to refill...")
time.sleep(3)

# Now we should have ~3 tokens again
for i in range(2):
    result = distributed_token_bucket("alice", max_tokens=5, refill_rate=1)
    status = "✅ ALLOWED" if result["allowed"] else "❌ DENIED"
    print(f"  Request {i+1}: {status}  (remaining: {result['remaining']})")

## 🧪 Proving It's Race-Condition Free

Let's run the same multi-threaded test, but this time using our Lua script. Since the script executes atomically, the race condition is impossible.

**Test setup:**
- 5 threads (simulating 5 servers)
- Each thread makes 5 requests (25 total)
- Bucket size: 10 tokens, no refill
- **Expected: exactly 10 allowed, 15 denied**

In [ ]:
import threading
import redis

r = redis.Redis(host="localhost", port=6381, decode_responses=True)

for key in r.keys("ratelimit:token:*"):
    r.delete(key)

print("=== Same Load, Atomic This Time ===")
print(f"Identical to the race demo: {GATEWAYS} gateways x {PER_GATEWAY} requests, "
      f"limit {LIMIT}.")
print("Only difference: read-check-write now happens inside one Lua script.\n")


def atomic_round() -> int:
    for key in r.keys("ratelimit:token:*"):
        r.delete(key)
    allowed = []
    bookkeeping = threading.Lock()

    def gateway():
        for _ in range(PER_GATEWAY):
            # refill_rate=0 → no tokens ever come back, so the answer must be
            # exactly the bucket size. Any refill would make this untestable.
            res = distributed_token_bucket("concurrent_test",
                                           max_tokens=LIMIT, refill_rate=0)
            with bookkeeping:
                allowed.append(res["allowed"])

    threads = [threading.Thread(target=gateway) for _ in range(GATEWAYS)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return sum(allowed)


rounds = [atomic_round() for _ in range(3)]
for i, n in enumerate(rounds, 1):
    mark = "✅ exact" if n == LIMIT else f"❌ off by {n - LIMIT}"
    print(f"  Round {i}: {n} allowed  {mark}")

assert all(n == LIMIT for n in rounds), f"Lua script was not atomic: {rounds}"
print(f"\n  Every round: exactly {LIMIT}. Not 'usually', not 'close enough'.")
print("  Redis is single-threaded for command execution, so a Lua script")
print("  holds the server for its whole duration. That is the guarantee.")
print("\n  ⚠️  The flip side: a slow script blocks *every* client. Keep rate")
print("     limiting scripts to a handful of O(1) commands — never loop over")
print("     a large key inside one.")

## ⏰ The Bug That's *Still* in Our Script

Atomicity fixed the race. But look again at how the script learns what time it is:

```lua
local now = tonumber(ARGV[3])   -- passed in by the caller
```

Each gateway sends **its own clock reading**. On one machine that's fine. Across a
fleet it is not: server clocks drift, NTP corrects in jumps, and a VM resuming from
a snapshot can be seconds or minutes off.

The script computes `elapsed = now - last_refill`. If gateway B's clock is 60 seconds
ahead of gateway A's, then B sees `elapsed = 60` and cheerfully refills the bucket to
full — **for every client**, on **every** request it handles. An attacker who can
influence a gateway's clock has an unlimited rate limit. Even without an attacker,
ordinary drift means your "100 req/min" limit is really "100 req/min, plus however
much clock skew you have".

Let's watch it happen.

In [ ]:
import redis
import time

r = redis.Redis(host="localhost", port=6381, decode_responses=True)
r.delete("ratelimit:token:skew_demo")

SKEW = 60.0   # gateway B's clock is one minute fast


def call_with_clock(client_id, clock, max_tokens=5, refill_rate=1.0):
    """The same Lua script, but we control what 'now' the gateway reports."""
    return r.evalsha(script_sha, 1, f"ratelimit:token:{client_id}",
                     max_tokens, refill_rate, clock)


print("Gateway A (correct clock) drains the bucket:")
for i in range(6):
    allowed, remaining = call_with_clock("skew_demo", time.time())
    print(f"  Request {i+1}: {'✅' if allowed else '❌'}  (tokens left: {remaining})")

print("\nGateway B (clock 60s fast) handles the very next request:")
allowed, remaining = call_with_clock("skew_demo", time.time() + SKEW)
print(f"  Request 7: {'✅ ALLOWED' if allowed else '❌ DENIED'}  (tokens left: {remaining})")

assert allowed, "expected the skewed clock to mint tokens"
print(f"""
  ⚠️  The bucket was empty. One request from a gateway whose clock is {SKEW:.0f}s
     fast refilled it completely. The limit was 5 per 5 seconds; this client
     just got 6 in under a second, and gateway B will keep doing it.

     The reverse is just as bad: a gateway with a SLOW clock writes a
     last_refill in the past, so the next correctly-clocked gateway computes a
     huge elapsed and refills the bucket for it.""")

### The fix: let Redis be the clock

Redis has a `TIME` command. Calling it **inside** the script means every gateway in
the fleet — however wrong its own clock is — shares one authoritative timeline. The
caller stops sending `now` at all, which also removes a parameter an attacker could
tamper with.

```lua
local t = redis.call('TIME')
local now = tonumber(t[1]) + tonumber(t[2]) / 1000000
```

> **The trade-off you're accepting.** `TIME` is non-deterministic, so the script can
> no longer be replicated verbatim to replicas — Redis replicates its *effects*
> instead (the default since Redis 5, so in practice this costs you nothing). More
> importantly, the timeline is now per-Redis-shard. If you shard rate limit keys
> across a cluster, two shards' clocks can still disagree with each other — but since
> a given client's bucket always lands on one shard, each bucket stays internally
> consistent. That is the property that actually matters.

In [ ]:
# Same algorithm, one line different: `now` comes from Redis, not the caller.
TOKEN_BUCKET_LUA_V2 = """
local key = KEYS[1]
local max_tokens = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])

-- Server-authoritative clock. No ARGV[3] any more.
local t = redis.call('TIME')
local now = tonumber(t[1]) + tonumber(t[2]) / 1000000

local bucket = redis.call('HMGET', key, 'tokens', 'last_refill')
local tokens = tonumber(bucket[1])
local last_refill = tonumber(bucket[2])

if tokens == nil then
    tokens = max_tokens
    last_refill = now
end

-- max(0, ...) is belt and braces: with a single clock source elapsed can never
-- be negative, but a defensive clamp costs nothing and prevents token minting.
local elapsed = math.max(0, now - last_refill)
tokens = math.min(max_tokens, tokens + elapsed * refill_rate)

local allowed = 0
if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
end

redis.call('HSET', key, 'tokens', tostring(tokens), 'last_refill', tostring(now))
redis.call('EXPIRE', key, 3600)

return {allowed, math.floor(tokens)}
"""

script_sha_v2 = r.script_load(TOKEN_BUCKET_LUA_V2)
r.delete("ratelimit:token:skew_demo_v2")


def call_v2(client_id, max_tokens=5, refill_rate=1.0):
    return r.evalsha(script_sha_v2, 1, f"ratelimit:token:{client_id}",
                     max_tokens, refill_rate)


print("Gateway A (correct clock) drains the bucket:")
for i in range(6):
    allowed, remaining = call_v2("skew_demo_v2")
    print(f"  Request {i+1}: {'✅' if allowed else '❌'}  (tokens left: {remaining})")

print("\nGateway B (clock 60s fast) — it no longer gets a say:")
allowed, remaining = call_v2("skew_demo_v2")
print(f"  Request 7: {'✅ ALLOWED' if allowed else '❌ DENIED'}  (tokens left: {remaining})")

assert not allowed, "skew fix failed — the bucket still refilled"
print("\n  ✅ Denied. The gateway's local clock is now irrelevant: it isn't an input.")
print("     This is the version that ships in app/server.py for Lab 4.")

## 🔍 Inspecting Redis State

Let's peek inside Redis to see what our rate limit data looks like. Each client gets a **hash** in Redis with two fields:
- `tokens` — how many tokens are left
- `last_refill` — timestamp of the last refill calculation

> 💡 Open **RedisInsight** at [http://localhost:5541](http://localhost:5541) to explore visually!

In [ ]:
import redis

r = redis.Redis(host="localhost", port=6381, decode_responses=True)

print("=== Rate Limit Keys in Redis ===\n")

# Find all rate limit keys
keys = sorted(r.keys("ratelimit:*"))
print(f"Found {len(keys)} rate limit keys:\n")

for key in keys:
    key_type = r.type(key)
    ttl = r.ttl(key)
    
    if key_type == "hash":
        data = r.hgetall(key)
        print(f"  🔑 {key}")
        print(f"     Type: hash | TTL: {ttl}s")
        for field, value in data.items():
            print(f"     {field}: {value}")
    elif key_type == "string":
        value = r.get(key)
        print(f"  🔑 {key}")
        print(f"     Type: string | TTL: {ttl}s | Value: {value}")
    print()

print("💡 Open RedisInsight at http://localhost:5541 to explore visually!")

## 🏭 Production Considerations

Our distributed token bucket works great for learning. Here are a few things you'd add for a real production system:

### 1. Connection Pooling
Don't create a new Redis connection for every request — reuse connections from a pool:
```python
pool = redis.ConnectionPool(host='localhost', port=6381, max_connections=20)
r = redis.Redis(connection_pool=pool)
```

### 2. Key Expiration
Always set `EXPIRE` on rate limit keys (we already do this!). Without it, keys for inactive users stay in memory forever — a **memory leak**.

### 3. Fail Open vs Fail Closed
What happens when Redis is down? You have two choices, and neither is free:
- **Fail open** — allow everything. Your API stays alive, but is unprotected for the
  duration of the incident. If the thing that killed Redis was a traffic surge, you
  have just removed the only defence against it.
- **Fail closed** — reject everything with 429. Nobody can abuse you, and nobody can
  use you either: a rate limiter outage becomes a full API outage.

**Default to fail open, and alert loudly.** Choose fail closed only when the work
behind the limiter costs more than the outage does — LLM inference, SMS sends,
payment processing. `app/server.py` in Lab 4 makes this an explicit
`RATE_LIMIT_FAIL_MODE` setting rather than an accident of which exception you caught.

A third option worth knowing: **fall back to a local in-process limiter** during the
outage. Each gateway enforces `global_limit / gateway_count` on its own. The limit is
sloppy — traffic is never distributed evenly — but it is far better than nothing and
it degrades instead of breaking.

### 4. Redis Cluster for High Scale
A single Redis instance handles ~100K operations/second. For 1M+ requests/sec, use **Redis Cluster** to shard rate limit keys across multiple Redis nodes.

In [ ]:
import redis

r = redis.Redis(host="localhost", port=6381, decode_responses=True)

# Clean up all rate limit keys
for key in r.keys("ratelimit:*"):
    r.delete(key)
print("✅ Cleaned up all rate limit keys from Redis")

## 📝 Key Takeaways

1. **Local rate limiters fail in distributed systems** — each server only sees its own traffic, so users can bypass limits by hitting different servers.

2. **Redis provides a fast shared store** that all servers can access — reads and writes take less than 1ms.

3. **Simple GET + SET has a race condition** (TOCTOU bug) — two servers can read the same counter value and both allow a request.

4. **Lua scripts run atomically in Redis** — the entire read-check-update happens as one uninterruptible operation, eliminating race conditions.

5. **Always set key expiration** to prevent memory leaks from inactive users.

6. **Production systems need extras** — connection pooling, failover strategy (fail open vs closed), and sharding for high scale.

## ⏭️ What's Next

In the next notebook, we'll build a complete **rate-limited API** with Flask that uses our distributed Redis token bucket to protect endpoints. We'll also add proper HTTP headers (`X-RateLimit-Remaining`, `Retry-After`) so clients know exactly how many requests they have left.

See you in **Lab 4**! 🚀